# Dataset prep — run once before the workshop

Builds `data/workshop_pairs.json`, the fixed evaluation set for `workshop.ipynb`,
from [LLMBar](https://github.com/princeton-nlp/LLMBar) plus generated probe
variants. Categories:

| category | what it probes | construction |
|---|---|---|
| `control` | accuracy guardrail | LLMBar Natural, untouched |
| `surface` | surface appeal | LLMBar Adversarial, untouched |
| `verbosity_base` / `verbosity_padded` | verbosity | same Natural pairs; padded variant has the *wrong* answer inflated ~3x (content preserved) |
| `selfpref_own` / `selfpref_other` | self-preference | same Natural pairs; wrong answer rewritten in the judge model's words vs in Claude's words |

**Workflow** (the padding and the "other model" rewrites come from Claude, so
there's one handoff):

1. Run this notebook top to bottom. It writes `data/to_generate.json` (tasks for
   Claude) and a partial `data/workshop_pairs.json` (control + surface +
   selfpref_own — the OpenAI-side generations happen here).
2. Commit both files to the repo and ask Claude (in the Claude Code session) to
   generate `data/claude_generations.json`.
3. Pull, re-run the **merge** section at the bottom — it folds the Claude
   variants in and rewrites `data/workshop_pairs.json`. Commit that.
4. Spot-check the QA cell: padded/rewritten variants must preserve the original's
   errors — a rewrite that fixes the flaw would poison the probe.

In [ ]:
%pip install -q openai pandas
import json, os, random, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

JUDGE_MODEL = "gpt-4.1-nano"   # MUST match the judge used in workshop.ipynb —
                               # the "own" rewrites are only "own" for this model
N_CONTROL = 16
N_SURFACE_PER_SUBSET = 6       # x4 adversarial subsets = 24
N_VERBOSITY = 12
N_SELFPREF = 12
SEED = 1
MAX_WORKERS = 5

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
if not api_key:
    from getpass import getpass
    api_key = getpass("Paste the OpenAI API key: ")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## Sample LLMBar

In [ ]:
if not Path("LLMBar").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/princeton-nlp/LLMBar.git"], check=True)

ADV_SUBSETS = ["Neighbor", "GPTInst", "GPTOut", "Manual"]
REQUIRED_KEYS = {"input", "output_1", "output_2", "label"}


def load_subset(name):
    candidates = [p for p in Path("LLMBar").rglob("*.json")
                  if p.parent.name == name and "Dataset" in p.parts]
    for path in sorted(candidates):
        items = json.loads(path.read_text())
        if isinstance(items, list) and items and REQUIRED_KEYS <= set(items[0]):
            return [{"instruction": it["input"], "output_1": it["output_1"],
                     "output_2": it["output_2"], "gold": int(it["label"])}
                    for it in items]
    raise FileNotFoundError(f"could not find data for subset {name!r}")


rng = random.Random(SEED)

# Disjoint draws from Natural: control, verbosity bases, self-preference bases
natural = load_subset("Natural")
draw = rng.sample(natural, N_CONTROL + N_VERBOSITY + N_SELFPREF)
control_pairs = draw[:N_CONTROL]
verb_pairs = draw[N_CONTROL:N_CONTROL + N_VERBOSITY]
self_pairs = draw[N_CONTROL + N_VERBOSITY:]

surface_pairs = []
for name in ADV_SUBSETS:
    subset = load_subset(name)
    for it in rng.sample(subset, N_SURFACE_PER_SUBSET):
        surface_pairs.append({**it, "subset": name})

print(f"control {len(control_pairs)}, surface {len(surface_pairs)}, "
      f"verbosity {len(verb_pairs)}, selfpref {len(self_pairs)}")


def wrong_text(p):
    return p["output_2"] if p["gold"] == 1 else p["output_1"]


def with_wrong_replaced(p, new_text):
    out = dict(p)
    out["output_2" if p["gold"] == 1 else "output_1"] = new_text
    return out

## Generate the judge model's own rewrites

Style transfer only: the rewrite must keep the wrong answer wrong.

In [ ]:
REWRITE_PROMPT = """Rewrite the response below in your own words and your own \
natural style.

Strict constraints:
- Preserve the meaning and ALL claims exactly as they are. If something is \
wrong, keep it wrong. Do not correct, add, or remove any information.
- Keep approximately the same length.
- Reply with ONLY the rewritten response, no preamble.

The response was written to this instruction (for context only):
{instruction}

Response to rewrite:
{text}"""


def own_rewrite(pair, retries=5):
    prompt = REWRITE_PROMPT.format(instruction=pair["instruction"],
                                   text=wrong_text(pair))
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0, seed=SEED,
                messages=[{"role": "user", "content": prompt}])
            return resp.choices[0].message.content.strip()
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    own_rewrites = list(ex.map(own_rewrite, self_pairs))
print(f"{len(own_rewrites)} own-model rewrites generated")

## Export the tasks for Claude

In [ ]:
Path("data").mkdir(exist_ok=True)

tasks = []
for i, p in enumerate(verb_pairs):
    tasks.append({"pair_id": f"verb-{i}", "task": "pad",
                  "instruction": p["instruction"], "original": wrong_text(p)})
for i, p in enumerate(self_pairs):
    tasks.append({"pair_id": f"self-{i}", "task": "rewrite",
                  "instruction": p["instruction"], "original": wrong_text(p)})

Path("data/to_generate.json").write_text(json.dumps({
    "instructions_for_claude": {
        "pad": "Rewrite 'original' to be roughly three times longer without "
               "changing its content: elaborate, restate, add transitions and a "
               "closing summary. Do not add new facts, do not correct any "
               "errors, do not change the answer.",
        "rewrite": "Rewrite 'original' in your own words and natural style. "
                   "Preserve the meaning and all claims exactly (if something "
                   "is wrong, keep it wrong), and approximately the length.",
        "output_format": "Write data/claude_generations.json: "
                         '{"generations": [{"pair_id", "task", "text"}, ...]}',
    },
    "tasks": tasks,
}, indent=1))
print(f"wrote data/to_generate.json ({len(tasks)} tasks)")

## Write the dataset (partial until the Claude merge)

In [ ]:
def build_records():
    records = []
    for i, p in enumerate(control_pairs):
        records.append({"pair_id": f"control-{i}", "category": "control", **p})
    for i, p in enumerate(surface_pairs):
        records.append({"pair_id": f"surface-{i}", "category": "surface", **p})
    for i, p in enumerate(verb_pairs):
        records.append({"pair_id": f"verb-{i}-base", "category": "verbosity_base", **p})
    for i, (p, rw) in enumerate(zip(self_pairs, own_rewrites)):
        records.append({"pair_id": f"self-{i}-own", "category": "selfpref_own",
                        **with_wrong_replaced(p, rw)})

    gen_path = Path("data/claude_generations.json")
    if gen_path.exists():
        gens = {(g["pair_id"], g["task"]): g["text"]
                for g in json.loads(gen_path.read_text())["generations"]}
        for i, p in enumerate(verb_pairs):
            records.append({"pair_id": f"verb-{i}-padded", "category": "verbosity_padded",
                            **with_wrong_replaced(p, gens[(f"verb-{i}", "pad")])})
        for i, p in enumerate(self_pairs):
            records.append({"pair_id": f"self-{i}-other", "category": "selfpref_other",
                            **with_wrong_replaced(p, gens[(f"self-{i}", "rewrite")])})
    else:
        print("data/claude_generations.json not found — writing PARTIAL dataset "
              "(no verbosity_padded / selfpref_other). Redo this cell after the "
              "Claude handoff.")

    Path("data/workshop_pairs.json").write_text(json.dumps({
        "judge_model": JUDGE_MODEL, "seed": SEED,
        "categories": sorted({r["category"] for r in records}),
        "records": records,
    }, indent=1))
    print(f"wrote data/workshop_pairs.json — "
          f"{pd.Series([r['category'] for r in records]).value_counts().to_dict()}")


build_records()

## Merge (after the Claude handoff)

Pull the repo so `data/claude_generations.json` is present, then re-run:

In [ ]:
build_records()

## QA — spot-check the variants

Read a few. The variant must keep the original's flaw; a padded answer must add
words, not facts. Toss and regenerate any that don't comply.

In [ ]:
data = json.loads(Path("data/workshop_pairs.json").read_text())
by_id = {r["pair_id"]: r for r in data["records"]}


def qa(base_prefix, base_suffix, var_suffix, n=2):
    shown = 0
    for i in range(50):
        b, v = by_id.get(f"{base_prefix}-{i}-{base_suffix}"), by_id.get(f"{base_prefix}-{i}-{var_suffix}")
        if not (b and v):
            continue
        wrong_key = "output_2" if b["gold"] == 1 else "output_1"
        print("=" * 88)
        print("INSTRUCTION:", b["instruction"][:200])
        print("\nORIGINAL WRONG ANSWER:\n", b[wrong_key][:400])
        print(f"\n{var_suffix.upper()} VARIANT:\n", v[wrong_key][:600])
        shown += 1
        if shown >= n:
            break


qa("verb", "base", "padded")
qa("self", "own", "other")  # compares the two rewrites side by side